In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")


ROOT = cari_akar_repo()
sys.path.insert(0, str(ROOT))
from src.common.manifest import sha256_raw      # lineage dari starter, bukan ditulis tangan

RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
GAMBAR = ROOT / "reports" / "gambar"
PROFIL.mkdir(parents=True, exist_ok=True)
GAMBAR.mkdir(parents=True, exist_ok=True)

# ---- parameter (dicatat di log; ubah hanya dengan alasan tertulis) ----
Z_ROBUST = 3.5        # ambang z-robust (MAD) untuk lonjakan tahun-ke-tahun / kuartal-ke-kuartal
MIN_SELISIH = 1.0     # selisih minimal (poin persen) agar ditandai, supaya deret yang sangat mulus tidak banjir tanda
MIN_REL = 0.15        # ambang selisih juga minimal 15% dari median deret (deret bertingkat tinggi wajar lebih bergejolak)
MIN_N = 8             # minimal observasi per deret untuk dihitung IQR dan lonjakan

BERKAS_ILO = {
    "SDG_0852_SEX_AGE_RT": "ilostat/2026-10-20/data/DF_SDG_0852_SEX_AGE_RT.csv",
    "SDG_0852_SEX_DSB_RT": "ilostat/2026-10-20/data/DF_SDG_0852_SEX_DSB_RT.csv",
    "UNE_DEAP_SEX_AGE_RT": "ilostat/2026-10-20/data/DF_UNE_DEAP_SEX_AGE_RT.csv",
}


def ke_indeks(tp, freq):
    """Periode ILO -> bilangan bulat berurutan. A: '2020'; Q: '2020-Q3'."""
    if freq == "A":
        return int(tp[:4])
    if freq == "Q":
        y, q = tp.split("-Q")
        return int(y) * 4 + int(q) - 1
    raise ValueError(f"FREQ {freq!r} belum ditangani; tambahkan aturannya")


def ke_label(i, freq):
    return str(i) if freq == "A" else f"{i // 4}-Q{i % 4 + 1}"


ringkas, celah_all, kandidat_all, dibaca = [], [], [], {}
for seri, rel in BERKAS_ILO.items():
    d = pd.read_csv(RAW / rel, dtype=str)
    d["_baris_csv"] = d.index + 2                      # header = baris 1
    d["_berkas_raw"], d["_sha256_raw"] = rel, sha256_raw(rel)
    dims = [c for c in ["SEX", "AGE", "DSB"] if c in d.columns]
    kunci = ["REF_AREA", "FREQ", "MEASURE", *dims, "TIME_PERIOD"]
    d["nilai"] = pd.to_numeric(d["OBS_VALUE"], errors="coerce")
    d["idx"] = [ke_indeks(t, f) for t, f in zip(d["TIME_PERIOD"], d["FREQ"])]
    for k in ["NOTE_INDICATOR", "OBS_STATUS", "SOURCE"]:
        d[k] = d[k].fillna("")
    d["catatan_patahan"] = d["NOTE_INDICATOR"].str.contains("Break in series")

    # ---------- 1. DUPLIKAT ----------
    n_kunci = int(d.duplicated(kunci, keep=False).sum())
    n_identik = int(d.duplicated([c for c in d.columns if not c.startswith("_") and c not in ("nilai", "idx", "catatan_patahan")], keep=False).sum())
    n_nilai_beda = int(d.groupby(kunci)["OBS_VALUE"].nunique().gt(1).sum())     # kunci sama, nilai berbeda = revisi

    # ---------- 2. MISSING ----------
    n_null = int(d["nilai"].isna().sum())
    grp = ["FREQ", *dims]
    dn = d[d["nilai"].notna()]                             # baris tanpa nilai tidak dihitung ada
    awal_seri = dn.groupby("FREQ")["idx"].min()
    akhir_seri = dn.groupby("FREQ")["idx"].max()
    for key, g in d.groupby(grp):
        key = key if isinstance(key, tuple) else (key,)
        f = key[0]
        ada = set(g.loc[g["nilai"].notna(), "idx"])
        if not ada:
            continue
        lo, hi = min(ada), max(ada)
        penuh = range(lo, hi + 1)
        hilang = [i for i in penuh if i not in ada]
        celah_all.append({"seri": seri, "FREQ": f, **dict(zip(dims, key[1:])),
                          "periode_pertama": ke_label(lo, f), "periode_terakhir": ke_label(hi, f),
                          "n_ada": len(ada), "n_diharapkan_dalam_rentang": len(penuh), "n_celah": len(hilang),
                          "celah": ";".join(ke_label(i, f) for i in hilang),
                          "n_sebelum_mulai": int(lo - awal_seri[f]), "n_setelah_akhir": int(akhir_seri[f] - hi)})

    # ---------- 3. OUTLIER (per deret rincian, bukan global) ----------
    for key, g in d.groupby(grp):
        key = key if isinstance(key, tuple) else (key,)
        f = key[0]
        g = g.dropna(subset=["nilai"]).sort_values("idx")
        if len(g) < MIN_N:
            continue
        v = g["nilai"]
        lag = 4 if f == "Q" else 1                          # kuartalan dibandingkan dengan kuartal yang sama tahun lalu (musiman)
        dv = (v - v.shift(lag)).where((g["idx"] - g["idx"].shift(lag)) == lag)
        if f == "Q":
            sisa = dv
        else:
            sisa = v - v.rolling(5, center=True, min_periods=3).median()   # tren panjang tidak ikut dianggap anomali
        q1, q3 = sisa.quantile(0.25), sisa.quantile(0.75)
        iqr = q3 - q1
        lo_b, hi_b = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        med = dv.median()
        mad = (dv - med).abs().median()
        z = 0.6745 * (dv - med) / mad if mad and mad > 0 else pd.Series(np.nan, index=g.index)
        ambang = max(MIN_SELISIH, MIN_REL * abs(v.median()))
        f_iqr = ((sisa < lo_b) | (sisa > hi_b)) & (sisa.abs() >= ambang)
        f_lon = (z.abs() >= Z_ROBUST) & (dv.abs() >= ambang)
        for ix in g.index[f_iqr | f_lon]:
            r = g.loc[ix]
            kandidat_all.append({
                "seri": seri,
                "FREQ": f,
                **dict(zip(dims, key[1:])),
                "TIME_PERIOD": r["TIME_PERIOD"],
                "nilai": r["nilai"],
                "metode": "+".join(m for m, ok in [("IQR_sisa", f_iqr[ix]), ("lonjakan", f_lon[ix])] if ok),
                "batas_bawah": round(lo_b, 2),
                "batas_atas": round(hi_b, 2),
                "selisih": round(dv[ix], 2) if pd.notna(dv[ix]) else np.nan,
                "sisa_dari_median_lokal": round(sisa[ix], 2),
                "z_robust": round(z[ix], 2) if pd.notna(z[ix]) else np.nan,
                "OBS_STATUS": r["OBS_STATUS"],
                "NOTE_INDICATOR": r["NOTE_INDICATOR"],
                "catatan_patahan": bool(r["catatan_patahan"]),
                "SOURCE": r["SOURCE"],
                "pembanding": "kuartal sama tahun lalu" if f == "Q" else "periode sebelumnya",
                "ambang": round(ambang, 2),
                "berkas_raw": r["_berkas_raw"],
                "baris_csv": int(r["_baris_csv"]),
                "sha256_raw": r["_sha256_raw"],
            })

    idx_min_row = d.loc[d["idx"].idxmin()]
    dibaca[seri] = d
    ringkas.append({
        "seri": seri,
        "baris_mentah": len(d),
        "n_deret_rincian": d.groupby(grp).ngroups,
        "frekuensi": ",".join(sorted(d["FREQ"].unique())),
        "periode_min": ke_label(int(idx_min_row["idx"]), idx_min_row["FREQ"]),
        "obs_kosong": n_null,
        "kunci_kembar": n_kunci,
        "baris_identik_penuh": n_identik,
        "kunci_nilai_berbeda": n_nilai_beda,
    })

celah = pd.DataFrame(celah_all)
kand = pd.DataFrame(kandidat_all)
ring = pd.DataFrame(ringkas)
ring["deret_dengan_celah"] = ring["seri"].map(celah[celah["n_celah"] > 0].groupby("seri").size()).fillna(0).astype(int)
ring["total_periode_celah"] = ring["seri"].map(celah.groupby("seri")["n_celah"].sum()).fillna(0).astype(int)
ring["calon_anomali"] = ring["seri"].map(kand.groupby("seri").size() if len(kand) else {}).fillna(0).astype(int)
ring["calon_bertepatan_patahan"] = ring["seri"].map(kand[kand["catatan_patahan"]].groupby("seri").size() if len(kand) else {}).fillna(0).astype(int)

ring.to_csv(PROFIL / "ilo_ringkasan_profil.csv", index=False)
celah.to_csv(PROFIL / "ilo_missing_celah_periode.csv", index=False)
kand.to_csv(PROFIL / "ilo_calon_anomali.csv", index=False)

print("=== Ringkasan profil ILO ===")
print(ring.to_string(index=False))
print("\n=== Deret dengan celah periode di dalam rentangnya (missing aktual) ===")
print(celah[celah["n_celah"] > 0].drop(columns=["n_sebelum_mulai", "n_setelah_akhir"]).to_string(index=False) if (celah["n_celah"] > 0).any() else "  tidak ada")
print("\n=== Deret yang mulai lebih lambat dari deret lain dalam seri yang sama (struktural, bukan missing) ===")
print(celah[celah["n_sebelum_mulai"] + celah["n_setelah_akhir"] > 0][["seri", "FREQ", "periode_pertama", "periode_terakhir", "n_sebelum_mulai", "n_setelah_akhir"]].to_string(index=False)
      if ((celah["n_sebelum_mulai"] + celah["n_setelah_akhir"]) > 0).any() else "  tidak ada")
print("\n=== Calon anomali ILO ===")
print(kand[["seri", "FREQ", "TIME_PERIOD", "nilai", "metode", "selisih", "catatan_patahan", "baris_csv"]].to_string(index=False) if len(kand) else "  tidak ada")

# ---------- grafik: deret tahunan total jenis kelamin, calon ditandai merah ----------
fig, axes = plt.subplots(len(BERKAS_ILO), 1, figsize=(11, 3.4 * len(BERKAS_ILO)), sharex=False)
for ax, (seri, d) in zip(axes, dibaca.items()):
    dims = [c for c in ["AGE", "DSB"] if c in d.columns]
    s = d[(d["FREQ"] == "A")]
    if "SEX" in s.columns:
        s = s[s["SEX"] == "SEX_T"]
    for key, g in s.groupby(dims):
        g = g.dropna(subset=["nilai"]).sort_values("idx")
        ax.plot(g["idx"], g["nilai"], marker="o", ms=2, lw=1, label=str(key).replace("AGE_", "").replace("DSB_", "")[:28])
    if len(kand):
        k = kand[(kand["seri"] == seri) & (kand["FREQ"] == "A")]
        if "SEX" in k.columns:
            k = k[k["SEX"] == "SEX_T"]
        k_idx = [ke_indeks(t, f) for t, f in zip(k["TIME_PERIOD"], k["FREQ"])]
        ax.scatter(k_idx, k["nilai"], c="red", zorder=5, s=30, label="calon anomali")
    for t in sorted(set(s.loc[s["catatan_patahan"], "idx"])):
        ax.axvline(t, color="orange", alpha=0.3)
    ax.set_title(f"{seri} (SEX_T, tahunan; garis oranye: catatan patahan ILO)", fontsize=9)
    ax.set_ylabel("%")
    ax.legend(fontsize=6, ncol=3)
fig.tight_layout()
fig.savefig(GAMBAR / "ilo_profil_deret.png", dpi=200)
plt.close(fig)

# ---------- rujukan BPS dari tugas #12/#13 (dibaca, tidak dihitung ulang) ----------
bps = {}
BERKAS_BPS = "bps/2026-10-17/sdg8_tpt.csv"
b = pd.read_csv(RAW / BERKAS_BPS)
bps["baris_mentah"] = len(b)
bps["kunci_kembar"] = int(b.duplicated(["wilayah_id", "turvar_id", "tahun", "periode"], keep=False).sum())
bps["baris_identik_penuh"] = int(b.duplicated(keep=False).sum())
bps["obs_kosong"] = int(pd.to_numeric(b["nilai"], errors="coerce").isna().sum())
f_status, f_anom = PROFIL / "status_sel_grid_sdg8_tpt.csv", PROFIL / "anomali_terklasifikasi_var543.csv"
if f_status.exists():
    st = pd.read_csv(f_status)["status"].value_counts().to_dict()
    bps.update({"sel_terisi": st.get("terisi", 0), "sel_struktural": st.get("kosong_struktural", 0), "sel_missing_aktual": st.get("missing_aktual", 0)})
if f_anom.exists():
    bps["calon_anomali"] = pd.read_csv(f_anom)["kategori"].value_counts().to_dict()
print("\n=== Rujukan BPS var543 ===")
print(bps)

# ---------- ringkasan markdown gabungan ----------
md = ["# Profiling missing, outlier, dan duplikat: seri utama BPS dan ILO", "",
      "Hanya menemukan dan menandai; tidak ada nilai di `data/raw/` yang diubah. Setiap calon dapat ditelusuri lewat "
      "`berkas_raw`, `baris_csv`, dan `sha256_raw` di berkas CSV masing-masing.", "",
      "## BPS var543 (rinci di tugas #12 dan #13)", ""]
md += [f"- {k}: {v}" for k, v in bps.items()]
md += ["", "## ILOSTAT", "",
       "| Seri | Baris | Deret rincian | Obs kosong | Kunci kembar | Identik penuh | Deret bercelah | Periode celah | Calon anomali | Bertepatan patahan |",
       "|---|---|---|---|---|---|---|---|---|---|"]
for r in ring.itertuples():
    md.append(f"| {r.seri} | {r.baris_mentah} | {r.n_deret_rincian} | {r.obs_kosong} | {r.kunci_kembar} | {r.baris_identik_penuh} | "
              f"{r.deret_dengan_celah} | {r.total_periode_celah} | {r.calon_anomali} | {r.calon_bertepatan_patahan} |")
md += ["", "Definisi: *celah* = periode yang hilang di antara periode pertama dan terakhir deret rincian itu (missing aktual). "
           "Periode sebelum deret mulai atau sesudah berakhir bukan missing (struktural). "
           "*Calon anomali* = sisa terhadap median lokal (5 periode; data kuartalan: selisih dengan kuartal sama tahun lalu) di luar 1,5 x IQR, "
           f"atau lonjakan dengan z-robust >= {Z_ROBUST}; keduanya hanya ditandai bila selisihnya >= max({MIN_SELISIH} poin, {int(MIN_REL * 100)}% dari median deret)."]
(PROFIL / "ringkasan_profiling_bps_ilo.md").write_text("\n".join(md) + "\n", encoding="utf-8")

METRIK_2 = {"ilo_obs_kosong": int(ring["obs_kosong"].sum()), "ilo_kunci_kembar": int(ring["kunci_kembar"].sum()),
            "ilo_periode_celah": int(ring["total_periode_celah"].sum()), "ilo_calon_anomali": int(ring["calon_anomali"].sum())}
print("\nAngka untuk log:", METRIK_2)

=== Ringkasan profil ILO ===
               seri  baris_mentah  n_deret_rincian frekuensi periode_min  obs_kosong  kunci_kembar  baris_identik_penuh  kunci_nilai_berbeda  deret_dengan_celah  total_periode_celah  calon_anomali  calon_bertepatan_patahan
SDG_0852_SEX_AGE_RT           216                9         A        2000           0             0                    0                    0                   0                    0              1                         0
SDG_0852_SEX_DSB_RT            81                9         A        2010           3             0                    0                    0                   9                   48              5                         0
UNE_DEAP_SEX_AGE_RT          5424              168       A,Q        1976         223             0                    0                    0                 167                 2946             96                        17

=== Deret dengan celah periode di dalam rentangnya (missing aktual) ===
      

In [8]:
# ---------- pola celah: serentak (periode tidak terbit) vs khusus rincian ----------
baris_pola = []
for (seri, f), g in celah.groupby(["seri", "FREQ"]):
    n_deret = len(g)
    hitung = {}
    for s in g["celah"].fillna(""):
        for p in (s.split(";") if s else []):
            hitung[p] = hitung.get(p, 0) + 1
    for p, n in hitung.items():
        baris_pola.append({"seri": seri, "FREQ": f, "periode": p, "deret_bercelah": n, "n_deret": n_deret, "porsi": round(n / n_deret, 2)})
pola = pd.DataFrame(baris_pola, columns=["seri", "FREQ", "periode", "deret_bercelah", "n_deret", "porsi"])
pola["pola"] = np.where(pola["porsi"] >= 0.9, "serentak_tidak_terbit", "khusus_rincian")
pola.to_csv(PROFIL / "ilo_pola_celah_periode.csv", index=False)

print("=== Pola celah per seri dan frekuensi ===")
print(pola.groupby(["seri", "FREQ", "pola"]).agg(periode_unik=("periode", "nunique"), sel_celah=("deret_bercelah", "sum")).reset_index().to_string(index=False))
print("\n=== Periode yang hilang serentak di (hampir) semua deret ===")
for (seri, f), g in pola[pola["pola"] == "serentak_tidak_terbit"].groupby(["seri", "FREQ"]):
    print(f"{seri} [{f}]:", ";".join(sorted(g["periode"])))
print("\n=== 10 deret dengan celah terbanyak ===")
print(celah.sort_values("n_celah", ascending=False).head(10)[["seri", "FREQ", "SEX", "AGE", "DSB", "periode_pertama", "periode_terakhir", "n_ada", "n_celah"]].to_string(index=False))

# ---------- observasi tanpa nilai (OBS_VALUE kosong) ----------
for seri, d in dibaca.items():
    k = d[d["nilai"].isna()]
    if len(k):
        print(f"\n=== {seri}: {len(k)} baris tanpa OBS_VALUE ===")
        print(k.groupby(["FREQ", "OBS_STATUS"]).size().to_string())
        print(k[["FREQ", *[c for c in ["SEX", "AGE", "DSB"] if c in k.columns], "TIME_PERIOD", "OBS_STATUS", "NOTE_INDICATOR"]].head(8).to_string(index=False))

# ---------- sebaran calon anomali ----------
if len(kand):
    print("\n=== Calon anomali per seri dan frekuensi ===")
    print(kand.groupby(["seri", "FREQ"]).size().to_string())
    print("\n=== Calon yang bertepatan dengan catatan patahan ILO, per tahun ===")
    print(kand[kand["catatan_patahan"]].groupby(["seri", "TIME_PERIOD"]).size().to_string() or "  tidak ada")
    print("\n=== Deret penyumbang calon terbanyak ===")
    sumbu = [c for c in ["seri", "FREQ", "SEX", "AGE", "DSB"] if c in kand.columns]
    print(kand.groupby(sumbu, dropna=False).size().sort_values(ascending=False).head(8).to_string())

=== Pola celah per seri dan frekuensi ===
               seri FREQ                  pola  periode_unik  sel_celah
SDG_0852_SEX_DSB_RT    A        khusus_rincian             3          3
SDG_0852_SEX_DSB_RT    A serentak_tidak_terbit             5         45
UNE_DEAP_SEX_AGE_RT    A        khusus_rincian            20        115
UNE_DEAP_SEX_AGE_RT    A serentak_tidak_terbit             1         80
UNE_DEAP_SEX_AGE_RT    Q        khusus_rincian            21        150
UNE_DEAP_SEX_AGE_RT    Q serentak_tidak_terbit            31       2601

=== Periode yang hilang serentak di (hampir) semua deret ===
SDG_0852_SEX_DSB_RT [A]: 2011;2012;2013;2014;2015
UNE_DEAP_SEX_AGE_RT [A]: 1995
UNE_DEAP_SEX_AGE_RT [Q]: 2008-Q2;2008-Q4;2009-Q2;2009-Q4;2010-Q2;2010-Q4;2011-Q2;2011-Q4;2012-Q2;2012-Q4;2013-Q2;2013-Q4;2014-Q2;2014-Q4;2015-Q2;2015-Q4;2016-Q2;2016-Q4;2017-Q2;2017-Q4;2018-Q2;2018-Q4;2019-Q2;2019-Q4;2020-Q2;2020-Q4;2021-Q2;2021-Q4;2022-Q2;2022-Q4;2023-Q2

=== 10 deret dengan celah terbanyak ==